# Cognifyz Machine Learning Internship

## Task 2 — Restaurant Recommendation System

### Objective

Create a restaurant recommendation system based on user preferences.

### Approach

A content-based filtering approach will be used to recommend
restaurants that are similar to the user's preferred criteria.

### Recommendation Criteria

- Cuisine preference
- Price range
- City
- Restaurant rating
- Online delivery
- Table booking

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv(
    "../outputs/results/cleaned_restaurant_dataset.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (9551, 22)


In [3]:
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes,Primary Cuisine
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Yes,No,No,No,3,4.8,Dark Green,Excellent,314,French
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Yes,No,No,No,3,4.5,Dark Green,Excellent,591,Japanese
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Yes,No,No,No,4,4.4,Green,Very Good,270,Seafood
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,No,No,No,No,4,4.9,Dark Green,Excellent,365,Japanese
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Yes,No,No,No,4,4.8,Dark Green,Excellent,229,Japanese


In [4]:
recommendation_columns = [
    "Restaurant Name",
    "City",
    "Locality",
    "Cuisines",
    "Price range",
    "Aggregate rating",
    "Has Online delivery",
    "Has Table booking"
]

df[recommendation_columns].head()

,Restaurant Name,City,Locality,Cuisines,Price range,Aggregate rating,Has Online delivery,Has Table booking
0,Le Petit Souffle,Makati City,"Century City Mall, Poblacion, Makati City","French, Japanese, Desserts",3,4.8,No,Yes
1,Izakaya Kikufuji,Makati City,"Little Tokyo, Legaspi Village, Makati City",Japanese,3,4.5,No,Yes
2,Heat - Edsa Shangri-La,Mandaluyong City,"Edsa Shangri-La, Ortigas, Mandaluyong City","Seafood, Asian, Filipino, Indian",4,4.4,No,Yes
3,Ooma,Mandaluyong City,"SM Megamall, Ortigas, Mandaluyong City","Japanese, Sushi",4,4.9,No,No
4,Sambo Kojin,Mandaluyong City,"SM Megamall, Ortigas, Mandaluyong City","Japanese, Korean",4,4.8,No,Yes


In [5]:
df[recommendation_columns].isnull().sum()

Restaurant Name        0
City                   0
Locality               0
Cuisines               0
Price range            0
Aggregate rating       0
Has Online delivery    0
Has Table booking      0
dtype: int64

In [6]:
rec_df = df[recommendation_columns].copy()

In [7]:
rec_df.head()

,Restaurant Name,City,Locality,Cuisines,Price range,Aggregate rating,Has Online delivery,Has Table booking
0,Le Petit Souffle,Makati City,"Century City Mall, Poblacion, Makati City","French, Japanese, Desserts",3,4.8,No,Yes
1,Izakaya Kikufuji,Makati City,"Little Tokyo, Legaspi Village, Makati City",Japanese,3,4.5,No,Yes
2,Heat - Edsa Shangri-La,Mandaluyong City,"Edsa Shangri-La, Ortigas, Mandaluyong City","Seafood, Asian, Filipino, Indian",4,4.4,No,Yes
3,Ooma,Mandaluyong City,"SM Megamall, Ortigas, Mandaluyong City","Japanese, Sushi",4,4.9,No,No
4,Sambo Kojin,Mandaluyong City,"SM Megamall, Ortigas, Mandaluyong City","Japanese, Korean",4,4.8,No,Yes


In [8]:
text_columns = [
    "City",
    "Locality",
    "Cuisines"
]

for col in text_columns:
    rec_df[col] = (
        rec_df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

In [9]:
rec_df[
    ["City", "Locality", "Cuisines"]
].head()

,City,Locality,Cuisines
0,makati city,"century city mall, poblacion, makati city","french, japanese, desserts"
1,makati city,"little tokyo, legaspi village, makati city",japanese
2,mandaluyong city,"edsa shangri-la, ortigas, mandaluyong city","seafood, asian, filipino, indian"
3,mandaluyong city,"sm megamall, ortigas, mandaluyong city","japanese, sushi"
4,mandaluyong city,"sm megamall, ortigas, mandaluyong city","japanese, korean"


In [10]:
rec_df["Price range"].value_counts().sort_index()

Price range
1    4444
2    3113
3    1408
4     586
Name: count, dtype: int64

In [11]:
rec_df["Aggregate rating"].describe()

count    9551.000000
mean        2.666370
std         1.516378
min         0.000000
25%         2.500000
50%         3.200000
75%         3.700000
max         4.900000
Name: Aggregate rating, dtype: float64

In [12]:
rec_df["content"] = (
    rec_df["Cuisines"] + " " +
    rec_df["Locality"] + " " +
    rec_df["City"]
)

In [13]:
rec_df[
    [
        "Restaurant Name",
        "City",
        "Locality",
        "Cuisines",
        "content"
    ]
].head()

,Restaurant Name,City,Locality,Cuisines,content
0,Le Petit Souffle,makati city,"century city mall, poblacion, makati city","french, japanese, desserts","french, japanese, desserts century city mall, ..."
1,Izakaya Kikufuji,makati city,"little tokyo, legaspi village, makati city",japanese,"japanese little tokyo, legaspi village, makati..."
2,Heat - Edsa Shangri-La,mandaluyong city,"edsa shangri-la, ortigas, mandaluyong city","seafood, asian, filipino, indian","seafood, asian, filipino, indian edsa shangri-..."
3,Ooma,mandaluyong city,"sm megamall, ortigas, mandaluyong city","japanese, sushi","japanese, sushi sm megamall, ortigas, mandaluy..."
4,Sambo Kojin,mandaluyong city,"sm megamall, ortigas, mandaluyong city","japanese, korean","japanese, korean sm megamall, ortigas, mandalu..."


In [14]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [15]:
tfidf_matrix = tfidf.fit_transform(
    rec_df["content"]
)

In [16]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9551, 1561)


In [17]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [18]:
print("Similarity matrix shape:", cosine_sim.shape)

Similarity matrix shape: (9551, 9551)


In [19]:
cosine_similarity(tfidf_matrix)

array([[1.        , 0.62525117, 0.15797053, ..., 0.        , 0.        ,
        0.        ],
       [0.62525117, 1.        , 0.1090209 , ..., 0.        , 0.        ,
        0.        ],
       [0.15797053, 0.1090209 , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.6816648 ,
        0.23577763],
       [0.        , 0.        , 0.        , ..., 0.6816648 , 1.        ,
        0.31657817],
       [0.        , 0.        , 0.        , ..., 0.23577763, 0.31657817,
        1.        ]], shape=(9551, 9551))

In [20]:
indices = pd.Series(
    rec_df.index,
    index=rec_df["Restaurant Name"].str.lower()
).drop_duplicates()

In [21]:
indices.head()

Restaurant Name
le petit souffle          0
izakaya kikufuji          1
heat - edsa shangri-la    2
ooma                      3
sambo kojin               4
dtype: int64

In [22]:
def recommend_similar_restaurants(
    restaurant_name,
    top_n=10
):
    restaurant_name = restaurant_name.lower()

    if restaurant_name not in indices:
        return pd.DataFrame()

    idx = indices[restaurant_name]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n + 1]

    restaurant_indices = [
        i[0] for i in similarity_scores
    ]

    result = rec_df.iloc[
        restaurant_indices
    ].copy()

    result["Similarity Score"] = [
        i[1] for i in similarity_scores
    ]

    return result

In [23]:
rec_df[
    "Restaurant Name"
].head(10)

0                            Le Petit Souffle
1                            Izakaya Kikufuji
2                      Heat - Edsa Shangri-La
3                                        Ooma
4                                 Sambo Kojin
5                                Din Tai Fung
6                                  Buffet 101
7                                     Vikings
8    Spiral - Sofitel Philippine Plaza Manila
9                                    Locavore
Name: Restaurant Name, dtype: str

In [24]:
recommend_similar_restaurants(
    "Restaurant X",
    top_n=5
)

""


In [25]:
user_preferences = {
    "city": "New Delhi",
    "cuisine": "North Indian",
    "price_range": 2,
    "min_rating": 4.0,
    "online_delivery": "Yes"
}

user_preferences

{'city': 'New Delhi',
 'cuisine': 'North Indian',
 'price_range': 2,
 'min_rating': 4.0,
 'online_delivery': 'Yes'}

In [26]:
filtered_df = rec_df.copy()

In [27]:
filtered_df = filtered_df[
    filtered_df["City"] == user_preferences["city"].lower()
]

In [28]:
filtered_df = filtered_df[
    filtered_df["Price range"] ==
    user_preferences["price_range"]
]

In [29]:
filtered_df = filtered_df[
    filtered_df["Aggregate rating"] >=
    user_preferences["min_rating"]
]

In [30]:
filtered_df = filtered_df[
    filtered_df["Has Online delivery"] ==
    user_preferences["online_delivery"]
]

In [31]:
print("Restaurants matching filters:", len(filtered_df))

Restaurants matching filters: 59


In [32]:
user_content = (
    user_preferences["cuisine"].lower() + " " +
    user_preferences["city"].lower()
)

In [33]:
user_vector = tfidf.transform(
    [user_content]
)

In [34]:
rec_df["Restaurant Name"].head(10)

0                            Le Petit Souffle
1                            Izakaya Kikufuji
2                      Heat - Edsa Shangri-La
3                                        Ooma
4                                 Sambo Kojin
5                                Din Tai Fung
6                                  Buffet 101
7                                     Vikings
8    Spiral - Sofitel Philippine Plaza Manila
9                                    Locavore
Name: Restaurant Name, dtype: str

In [35]:
recommend_similar_restaurants(
    "fLe Petit Soufle",
    top_n=5
)

""


In [36]:
recommendations = recommend_similar_restaurants(
    "fLe Petit Soufle",
    top_n=5
)

recommendations

""


# Day 7 — Task 2 Progress

## Work Completed

- Loaded the cleaned restaurant dataset.
- Selected features relevant to restaurant recommendations.
- Cleaned categorical/text features.
- Created a combined content representation using cuisine,
  locality, and city.
- Applied TF-IDF vectorization to represent restaurant content.
- Calculated cosine similarity between restaurant profiles.
- Created a basic similarity-based recommendation function.
- Defined sample user preferences for the final recommendation system.

## Recommendation Approach

The planned recommendation system uses a content-based filtering
approach. User preferences will first be used to filter suitable
restaurants, followed by similarity-based ranking.

## Next Step

The recommendation logic will be converted into a reusable function
that accepts user preferences and returns the top recommended
restaurants.